In [1]:
import andes
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

MODEL_A = 'andes_TDS_2035_base.xlsx'
MODEL_B = 'andes_TDS_2035_droop.xlsx'

### Time Domain Simulation

### A

In [2]:
ss_A = andes.main.load(MODEL_A, setup=False, config_path = "config_file.rc")
ss_A.setup()
ss_A.PFlow.run()

ss_A.TDS.config.tstep = 0.025
ss_A.TDS.config.tf = 2
ss_A.TDS.run()
ss_A.PQ.Ppf.v[14] += 0.5 # Add x*2000 MW load to a bus
ss_A.TDS.config.tf = 20
ss_A.TDS.run()


  0%|                                                    | 0/100 [00:00<?, ?%/s]

  0%|                                                    | 0/100 [00:00<?, ?%/s]

True

### B

In [ ]:
ss_B = andes.main.load(MODEL_B, setup=False, config_path = "config_file.rc")
ss_B.setup()
ss_B.PFlow.run()

ss_B.TDS.config.tstep = 0.025
ss_B.TDS.config.tf = 2
ss_B.TDS.run()
ss_B.PQ.Ppf.v[14] += 0.5 # Add x*2000 MW load to a bus
ss_B.TDS.config.tf = 20
ss_B.TDS.run()

  0%|                                                    | 0/100 [00:00<?, ?%/s]

  0%|                                                    | 0/100 [00:00<?, ?%/s]

True

### Frequency response

In [ ]:
gen_df_A = pd.read_excel(MODEL_A, sheet_name='GENROU')
df_A = ss_A.dae.ts.df
f_df_A = df_A.filter(regex=r"^f BusFreq")
time_A = df_A.index
bus_numbers_A = f_df_A.columns.str.extract(r'(\d+)').astype(int)[0]
gen_df_A["weight"] = gen_df_A["Sn"] * (gen_df_A["M"]/2)
bus_weights_A = gen_df_A.groupby("bus")["weight"].sum()
weights_A = bus_numbers_A.map(bus_weights_A).fillna(0.0).to_numpy()
f_COI_A = (f_df_A.to_numpy() @ weights_A) / weights_A.sum()


gen_df_B = pd.read_excel(MODEL_B, sheet_name='GENROU')
df_B = ss_B.dae.ts.df
f_df_B = df_B.filter(regex=r"^f BusFreq")
time_B = df_B.index
bus_numbers_B = f_df_B.columns.str.extract(r'(\d+)').astype(int)[0]
gen_df_B["weight"] = gen_df_B["Sn"] * (gen_df_B["M"]/2)
bus_weights_B = gen_df_B.groupby("bus")["weight"].sum()
weights_B = bus_numbers_B.map(bus_weights_B).fillna(0.0).to_numpy()
f_COI_B = (f_df_B.to_numpy() @ weights_B) / weights_B.sum()


plt.figure(figsize=(10, 6))
for col in f_df_A.columns:
    plt.plot(time_A, 50*f_df_A[col], color='grey')
for col in f_df_B.columns:
    plt.plot(time_B, 50*f_df_B[col], color='grey')
plt.plot(time_A, 50*f_COI_A, label='All Following', color='blue')
plt.plot(time_B, 50*f_COI_B, label='Some Forming', color='red')
plt.xlabel("Time [s]")
plt.ylabel("Frequency [Hz]")
plt.title("Frequency response")
plt.legend()
plt.grid(True)
#plt.savefig('FrequencyResponse.png')
plt.show()


### Power change

In [5]:
df_B = ss_B.dae.ts.df
P_df_B = df_B.filter(regex=r"^Pe REGF1")
time_B = df_B.index

Sb_B = ss_B.config.mva

plt.figure(figsize=(10, 6))
for col in P_df_B.columns:
    plt.plot(time_B, Sb_B*P_df_B[col], label=col)
    ls = list(Sb_B*P_df_B[col])
    lim = 1.1*ls[0]
    plt.plot([0,20],[lim,lim], color='grey', linestyle='dashed')
plt.xlabel("Time [s]")
plt.ylabel("Active Power [MW]")
plt.title("FFR active power response")
plt.legend()
plt.grid(True)
#plt.savefig('P_Response.png')
plt.show()